## Why bother with the Batch API?

The regular Gemini API is **synchronous**.  
Send request --> Wait --> Get a single response  
This is fine for one or a few calls.

The **Batch API** is different:
- You submit ALL your requests at once
- Gemini processes them in the background (asynchronously)
- You come back later to collect the results

**Why use it?**
- 50% cheaper than the regular API
- No rate limit pressure - Gemini handles the pacing
- Perfect for large, non-urgent tasks like this one

In [ ]:
import time
from google import genai
import pandas as pd, matplotlib.pyplot as plt

In [ ]:
#Setting API key as environment variable
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

### Getting the data

In [ ]:
df.to_csv("product_reviews.csv", index=False)

In [ ]:
reviews_data = [
    {"review_id": "R001", "review": "Absolutely love this product! Works perfectly and arrived fast."},
    {"review_id": "R002", "review": "Terrible quality. Broke after two days. Total waste of money."},
    {"review_id": "R003", "review": "It's okay. Does what it says, nothing special."},
    {"review_id": "R004", "review": "Great value for the price. Would definitely buy again."},
    {"review_id": "R005", "review": "The color was different from the photo. A bit disappointed."},
    {"review_id": "R006", "review": "Exactly as described. Happy with the purchase."},
    {"review_id": "R007", "review": "Stopped working after a week. Very frustrating."},
    {"review_id": "R008", "review": "Average product. Not bad, not great."},
    {"review_id": "R009", "review": "Exceeded my expectations! Highly recommend."},
    {"review_id": "R010", "review": "Packaging was damaged but the product itself was fine."},
]

df = pd.DataFrame(reviews_data)
df.head()

,review_id,review
0,R001,Absolutely love this product! Works perfectly ...
1,R002,Terrible quality. Broke after two days. Total ...
2,R003,"It's okay. Does what it says, nothing special."
3,R004,Great value for the price. Would definitely bu...
4,R005,The color was different from the photo. A bit ...


### Classify single review & setup

In [ ]:
review1 = df.review[0]
print(review1)

Absolutely love this product! Works perfectly and arrived fast.


In [ ]:
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash' # Fast and perfect for classification

In [ ]:
system_prompt = '''You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.'''

In [ ]:
combined_prompt = f"{system_prompt}\n\nReview: {review1}"
combined_prompt

'You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.\n\nReview: Absolutely love this product! Works perfectly and arrived fast.'

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=combined_prompt
)

In [ ]:
print("Sentiment: ",response.text)

Sentiment:  Positive


### Batch Processing - Small inline batch
##### Let's format 1 review

In [ ]:
review1

'Absolutely love this product! Works perfectly and arrived fast.'

In [ ]:
system_prompt

'You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.'

#### Required format

In [ ]:
row_dict = dict()
row_dict['contents'] = {
    "role": "user",
    "parts":[{"text": f"{system_prompt}\n\nReview: {review1}"}]
}

In [ ]:
row_dict = dict()
combined_prompt = f"{system_prompt}\n\nReview: {review1}"
row_dict['contents'] = {
    "parts":[{
        "text": combined_prompt
        }]
    }

In [ ]:
row_dict = dict()
combined_prompt = f"{system_prompt}\n\nReview: {review1}"
row_dict['contents'] = {'parts':[{'text': combined_prompt}]}

In [ ]:
row_dict

{'contents': {'parts': [{'text': 'You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.\n\nReview: Absolutely love this product! Works perfectly and arrived fast.'}]}}

#### Making a function to apply to all the reviews

In [ ]:
#Making a function
def format_review(review):
  combined_prompt = f"{system_prompt}\n\nReview: {review}"

  row_dict = dict()
  row_dict['contents'] = {
      "parts":[{"text": combined_prompt}]
  }
  return row_dict

In [ ]:
# def format_review(review):
#   row_dict = dict()
#   row_dict['config'] = {'system_instruction': system_prompt}
#   row_dict['contents'] = {"role": "user",
#                           "parts":[{"text": review1}]
#                           }
#   return row_dict

In [ ]:
tiny_batch_raw = df.review[:4]

In [ ]:
inline_requests = []
for rev in tiny_batch_raw:
  inline_requests.append(format_review(rev))

In [ ]:
inline_requests[:2]

[{'contents': {'parts': [{'text': 'You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.\n\nReview: Absolutely love this product! Works perfectly and arrived fast.'}]}},
 {'contents': {'parts': [{'text': 'You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.\n\nReview: Terrible quality. Broke after two days. Total waste of money.'}]}}]

#### Submit the batch and create the job

In [ ]:
inline_job = client.batches.create(
    model=MODEL_ID,
    src=inline_requests
)

In [ ]:
print(inline_job.name)

batches/xw62trax11hpf3peqcxvxhcz7mnjtgu3mf34


In [ ]:
# print(f"Submitted Inline Batch Job: {inline_job.name}")
# Submitted Inline Batch Job: batches/jibxsj5sl46gp2s75k1ck9ld4pyvs1d4g5y3

Submitted Inline Batch Job: batches/jibxsj5sl46gp2s75k1ck9ld4pyvs1d4g5y3


In [ ]:
job_info = client.batches.get(name="batches/xw62trax11hpf3peqcxvxhcz7mnjtgu3mf34")
job_info.state

<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'>

In [ ]:
# job_info = client.batches.get(name=inline_job.name)
job_info = client.batches.get(name="batches/xw62trax11hpf3peqcxvxhcz7mnjtgu3mf34")
job_info.state

<JobState.JOB_STATE_RUNNING: 'JOB_STATE_RUNNING'>

In [ ]:
job_info.state

<JobState.JOB_STATE_RUNNING: 'JOB_STATE_RUNNING'>

In [ ]:
while job_info.state not in ['SUCCEEDED', 'JOB_STATE_SUCCEEDED', 'FAILED', 'JOB_STATE_FAILED']:
    print(f"Status: {job_info.state}... waiting 30 seconds.")
    time.sleep(30)
    job_info = client.batches.get(name=inline_job.name)

Status: JobState.JOB_STATE_PENDING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_RUNNING... waiting 30 seconds.
Status: JobState.JOB_STATE_SUCCEEDED... waiting 30 seconds.
Status: JobState.JOB_STATE_SUCCEEDED... waiting 30 seconds.


KeyboardInterrupt: 

In [ ]:
while job_info.state not in ['SUCCEEDED', 'JOB_STATE_SUCCEEDED', 'FAILED', 'JOB_STATE_FAILED']:
    print(f"Status: {job_info.state}... waiting 30 seconds.")
    time.sleep(30)
    job_info = client.batches.get(name=inline_job.name)

#### Checking out the responses

In [ ]:
job_info.dest.inlined_responses

AttributeError: 'NoneType' object has no attribute 'inlined_responses'

In [ ]:
job_info.dest.inlined_responses[1].response.text

'Negative'

#### Getting the review and the sentiment into a dataframe

In [ ]:
# sentiment_df = pd.DataFrame(columns=['Review Text', 'Sentiment']) #creating a new dataframe

In [ ]:
sentiment_df = pd.DataFrame({'raw_object': job_info.dest.inlined_responses})
sentiment_df['original_review'] = tiny_batch_raw
sentiment_df['predicted_sentiment'] = sentiment_df['raw_object'].apply(lambda obj: obj.response.text.strip())

In [ ]:
# sentiment_df = pd.DataFrame(columns=['Review Text', 'Sentiment'])

# for i, item in enumerate(job_info.dest.inlined_responses):
#   original_review = tiny_batch_raw[i]
#   sentiment = item.response.text.strip()
#   sentiment_df.loc[i,["Review Text", "Sentiment"]] = [original_review, sentiment]

In [ ]:
sentiment_df

,raw_object,original_review,predicted_sentiment
0,response=GenerateContentResponse(\n candidate...,Absolutely love this product! Works perfectly ...,Positive
1,response=GenerateContentResponse(\n candidate...,Terrible quality. Broke after two days. Total ...,Negative
2,response=GenerateContentResponse(\n candidate...,"It's okay. Does what it says, nothing special.",Neutral
3,response=GenerateContentResponse(\n candidate...,Great value for the price. Would definitely bu...,Positive


## Batch Processing - Full batch

**JSONL** = JSON Lines. It's a simple format where:
- Each line is one complete JSON object
- Lines are separated by newlines
- It's easy to read, write, and stream line by line

### Step 1: creating the JSONL

When transitioning from inline batching to file-based batching (JSONL), there is one new idea: the **key** parameter.

Asynchronous file batches don't guarantee that the output will come back in the same order as the input. We must therefore assign a unique key to every single request. This allows us to accurately map the model's prediction back to the original review.

In [ ]:
def format_review_jsonl(ind, review):

  row_dict = dict()
  row_dict["key"] = str(ind)
  row_dict['request'] = format_review(review)

  return row_dict

In [ ]:
df[:4]

,review_id,review
0,R001,Absolutely love this product! Works perfectly ...
1,R002,Terrible quality. Broke after two days. Total ...
2,R003,"It's okay. Does what it says, nothing special."
3,R004,Great value for the price. Would definitely bu...


In [ ]:
for ind, _, review in df[:3].itertuples():
  print(format_review_jsonl(ind, review))

{'key': '0', 'request': {'contents': {'parts': [{'text': 'You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.\n\nReview: Absolutely love this product! Works perfectly and arrived fast.'}]}}}
{'key': '1', 'request': {'contents': {'parts': [{'text': 'You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.\n\nReview: Terrible quality. Broke after two days. Total waste of money.'}]}}}
{'key': '2', 'request': {'contents': {'parts': [{'text': "You are a sentiment analyzer. Classify the following review as Positive, Negative, or Neutral. Output only the label.\n\nReview: It's okay. Does what it says, nothing special."}]}}}


In [ ]:
import json
jsonl_filename = "full_batch.jsonl"

with open(jsonl_filename, "w") as file:
  for ind, _, review in df.itertuples():
    request_line = (format_review_jsonl(ind, review))
    file.write(json.dumps(request_line) + "\n")

### Step 2: Upload file to Gemini File API

In [ ]:
uploaded_file = client.files.upload(
    file=jsonl_filename,
    config={'display_name': 'sentiment_batch_input_10',
            'mime_type': 'application/jsonl'}
)

print("File uploaded successfully as: ",uploaded_file.name)

File uploaded successfully as:  files/hod3vihhz5bj


### Step 3: Create the batch job and wait

In [ ]:
file_batch_job = client.batches.create(
    model=MODEL_ID,
    src=uploaded_file.name,
    config={'display_name': 'Sentiment Analysis Job'}
)

print(file_batch_job.name)

batches/wq0rh5qj5jcez3y841brxpifphxei9ji3uyy


In [ ]:
#batches/nkjxqpagslwgy6o8vnpo2hvamwcyvaf85gs8
print(file_batch_job.name)

batches/wq0rh5qj5jcez3y841brxpifphxei9ji3uyy


In [ ]:
# job_info = client.batches.get(name=file_batch_job.name)
job_info = client.batches.get(name='batches/nkjxqpagslwgy6o8vnpo2hvamwcyvaf85gs8')
job_info.state

<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'>

In [ ]:
while job_info.state not in ['SUCCEEDED', 'JOB_STATE_SUCCEEDED', 'FAILED', 'JOB_STATE_FAILED']:
    print(f"Status: {job_info.state}... waiting 30 seconds.")
    time.sleep(30)
    job_info = client.batches.get(name=file_batch_job.name)

In [ ]:
job_info

BatchJob(
  create_time=datetime.datetime(2026, 5, 11, 10, 22, 44, 43698, tzinfo=TzInfo(0)),
  dest=BatchJobDestination(
    file_name='files/batch-nkjxqpagslwgy6o8vnpo2hvamwcyvaf85gs8'
  ),
  display_name='Sentiment Analysis Job',
  end_time=datetime.datetime(2026, 5, 11, 11, 29, 3, 308731, tzinfo=TzInfo(0)),
  model='models/gemini-2.5-flash',
  name='batches/nkjxqpagslwgy6o8vnpo2hvamwcyvaf85gs8',
  state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'>,
  update_time=datetime.datetime(2026, 5, 11, 11, 29, 3, 308731, tzinfo=TzInfo(0))
)

### Step 4: Download the results file

In [ ]:
result_bytes = client.files.download(file=job_info.dest.file_name)

In [ ]:
# Option 1: Save to file, read it, extract info, save to dataframe
with open("batch_results.jsonl", "wb") as f:
        f.write(result_bytes)

with open("batch_results.jsonl", "r") as f:
        first_result = json.loads(f.readline())
        key = first_result.get("key")
        sentiment = first_result["response"]["candidates"][0]["content"]["parts"][0]["text"].strip()

        print(f"Key ID: {key}")
        print(f"Predicted Sentiment: {sentiment}")

Key ID: 0
Predicted Sentiment: Positive


In [ ]:
import io

In [ ]:
# Option 2 - read into dataframe, extract deeply nested value into new column
res_df = pd.read_json(io.BytesIO(result_bytes), lines=True)

In [ ]:
res_df.response[0]['candidates'][0]['content']['parts'][0]['text']

'Positive'

In [ ]:
def extract_sentiment(resp):
  return resp['candidates'][0]['content']['parts'][0]['text'].strip()

In [ ]:
res_df['sentiment'] = res_df['response'].apply(extract_sentiment)

In [ ]:
# res_df['sentiment'] = res_df['response'].apply(
#         lambda resp: resp['candidates'][0]['content']['parts'][0]['text'].strip()
#     )

In [ ]:
res_df.head()

,response,key,sentiment
0,"{'responseId': 'B7YBasHUPKynqtsPuM3IsQI', 'can...",0,Positive
1,"{'responseId': 'CLYBatjFJcrQz7IP3oq4oQw', 'can...",1,Negative
2,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",2,Neutral
3,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",3,Positive
4,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",4,Negative


In [ ]:
res_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   response   10 non-null     object
 1   key        10 non-null     int64 
 2   sentiment  10 non-null     object
dtypes: int64(1), object(2)
memory usage: 372.0+ bytes


In [ ]:
res_df

,response,key,sentiment
0,"{'responseId': 'B7YBasHUPKynqtsPuM3IsQI', 'can...",0,Positive
1,"{'responseId': 'CLYBatjFJcrQz7IP3oq4oQw', 'can...",1,Negative
2,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",2,Neutral
3,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",3,Positive
4,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",4,Negative
5,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",5,Positive
6,"{'responseId': 'B7YBarXHMc-UmtkP-8LjqAM', 'can...",6,Negative
7,"{'responseId': 'CLYBar-SGc_Uz7IPwfT5sAw', 'can...",7,Neutral
8,"{'responseId': 'f70BaqvgCtiS_uMPztLhwAI', 'can...",8,Positive
9,"{'modelVersion': 'gemini-2.5-flash', 'usageMet...",9,Positive
